# Experiment: Count Effective Time Steps In `data`

Objective:
- Read `loc_data_kuramoto/generated_data.npz` inside this project.
- Focus only on the `data` array.
- Count how many time steps are non-`NaN` for each start point, without counting the last feature dimension separately.


In [11]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "loc_data_kuramoto").exists():
    candidate = project_root / "causal_network_mix_2_0.2_syn2_theta"
    if candidate.exists():
        project_root = candidate

npz_path = project_root / "loc_data_kuramoto" / "generated_data.npz"
print(f"project_root: {project_root}")
print(f"npz_path: {npz_path}")


project_root: /home/wangzhipeng/code/causal_network/causal_network_mix_2_0.2_syn2_theta
npz_path: /home/wangzhipeng/code/causal_network/causal_network_mix_2_0.2_syn2_theta/loc_data_kuramoto/generated_data.npz


## Plan

- Step 1: load the `data` array from the `.npz` archive.
- Step 2: collapse the last feature dimension and count valid `(start_index, time_step)` pairs.
- Step 3: summarize valid time-step counts per start point and the overall total.


In [12]:
def load_npz_archive(path: Path) -> Dict[str, np.ndarray]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"npz file not found: {path}")

    with np.load(path, allow_pickle=False) as archive:
        keys = list(archive.keys())
        if not keys:
            raise ValueError(f"No arrays found in npz file: {path}")
        return {key: np.asarray(archive[key]) for key in keys}


def load_data_array(path: Path, key: str = "data") -> np.ndarray:
    archive = load_npz_archive(path)
    if key not in archive:
        raise KeyError(f"Key '{key}' not found in {path}. Available keys: {tuple(archive.keys())}")

    data = np.asarray(archive[key], dtype=np.float32)
    if data.ndim != 3:
        raise ValueError(f"Expected '{key}' to have shape [start, time, feature], but got {data.shape}")
    return data


def summarize_data_valid_time_steps(data: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    data = np.asarray(data, dtype=np.float32)
    non_nan_mask = ~np.isnan(data)

    # Ignore the last dimension: count one valid time step for a start point
    # if any feature at [start_index, time_step, :] is observed.
    valid_time_step_mask = np.any(non_nan_mask, axis=-1)
    fully_observed_mask = np.all(non_nan_mask, axis=-1)
    partial_nan_mask = valid_time_step_mask & ~fully_observed_mask

    valid_time_steps = valid_time_step_mask.sum(axis=1).astype(np.int32)
    total_possible_time_steps = int(data.shape[0] * data.shape[1])
    total_valid_time_steps = int(valid_time_step_mask.sum())

    per_start_df = pd.DataFrame(
        {
            "start_index": np.arange(data.shape[0], dtype=np.int32),
            "valid_time_steps": valid_time_steps,
            "padded_time_steps": np.full(data.shape[0], data.shape[1], dtype=np.int32),
            "valid_ratio": valid_time_steps / max(int(data.shape[1]), 1),
        }
    )

    summary_df = pd.DataFrame(
        [
            {
                "num_start_points": int(data.shape[0]),
                "padded_time_steps": int(data.shape[1]),
                "feature_dim": int(data.shape[2]),
                "total_valid_time_steps": total_valid_time_steps,
                "total_possible_time_steps": total_possible_time_steps,
                "valid_time_step_ratio": total_valid_time_steps / max(total_possible_time_steps, 1),
                "partial_nan_time_steps": int(partial_nan_mask.sum()),
            }
        ]
    )
    return per_start_df, summary_df, valid_time_step_mask


In [13]:
data = load_data_array(npz_path, key="data")
per_start_df, summary_df, valid_time_step_mask = summarize_data_valid_time_steps(data)

print("data shape:", data.shape)
display(summary_df)
display(per_start_df["valid_time_steps"].describe().to_frame().T)
print("valid_time_step_mask shape:", valid_time_step_mask.shape)
print("total valid (start_index, time_step) pairs:", int(valid_time_step_mask.sum()))


data shape: (10000, 1001, 32)


,num_start_points,padded_time_steps,feature_dim,total_valid_time_steps,total_possible_time_steps,valid_time_step_ratio,partial_nan_time_steps
0,10000,1001,32,237354,10010000,0.023712,0


,count,mean,std,min,25%,50%,75%,max
valid_time_steps,10000.0,23.7354,32.185009,1.0,7.0,16.0,32.0,1001.0


valid_time_step_mask shape: (10000, 1001)
total valid (start_index, time_step) pairs: 237354


In [10]:
1291 / 50 * 10000

258200.0

In [8]:
display(per_start_df)

# If you only need the scalar answer, use this value:
effective_time_step_count = int(valid_time_step_mask.sum())
effective_time_step_count


,start_index,valid_time_steps,padded_time_steps,valid_ratio
0,0,7,108,0.064815
1,1,10,108,0.092593
2,2,57,108,0.527778
3,3,16,108,0.148148
4,4,43,108,0.398148
5,5,6,108,0.055556
6,6,8,108,0.074074
7,7,15,108,0.138889
8,8,37,108,0.342593
9,9,41,108,0.379630


1291

## Notes

- `effective_time_step_count` counts valid `(start_index, time_step)` pairs and does not multiply by the feature dimension.
- `per_start_df` shows how many valid time steps each start point contributes.
